## Setup

Runs on Colab, no-op locally. Run it first, before anything else.

`torch` and `numpy` are deliberately left alone: Colab's builds are CUDA-matched, and replacing them costs minutes and forces a runtime restart.

In [ ]:
# --- Colab setup ---------------------------------------------------
# Installs only what Colab is missing. Locally this whole cell is skipped.
import subprocess, sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers==5.7.0",
            "datasets>=5.0.1",
            "umap-learn>=0.5.12",
            "plotly>=6.9.0",
        ],
        check=True,
    )
    print("Colab: dependencies installed.")
else:
    print("Local environment: nothing to install.")


# Preparations

**Run this before the workshop.** It downloads about 620 MB. A hundred people doing that at
once on conference wifi is how the first exercise loses its slot.

You do not need a local environment: Colab is enough. If you do want one, the instructions
are in the repository README under
[Environment Setup](https://github.com/carlomarxdk/workshop-transformers#environment-setup).
This notebook does not repeat them; it checks that they worked.

The cells below:

1. Import every package the workshop uses and print its version.
2. Download the ~600 MB language model used in the first half.
3. Download the true/false statement datasets used in notebook a3.
4. Download and load the small event model and data extract used in the second half.

If every cell runs without error, you are ready.

## 1. Check the packages

In [ ]:
import pandas as pd
import numpy as np
import transformers
import torch
import umap
import sklearn
import datasets

print("All libraries imported successfully.")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("UMAP:", umap.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Datasets:", datasets.__version__)

## 2. Preload the language model and data

Running the two cells below downloads everything ahead of time, so you are not waiting on
the conference wifi.

First the language model. Any of the three works: they differ only in how much disk space
and memory they take. `ModernBERT-base` is the one the workshop uses (if you want to experiment, or
have limited disk space, either of the others will do).

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_id = "answerdotai/ModernBERT-base"
# model_id = "answerdotai/ModernBERT-large"  # alternative (LARGER) model
# model_id = "bert-base-uncased"             # alternative (SMALLER) model

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForMaskedLM.from_pretrained(model_id).eval()

print(f"Loaded {model_id}")
print(f"  {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"  {len(tokenizer):,} tokens in the vocabulary")

And now the data: true and false statements, used in notebook a3.

In [ ]:
from datasets import load_dataset

ds = load_dataset("carlomarxx/trilemma-of-truth", "city_locations")
assert ds["train"].shape == (3999, 11)
print("Loaded city_locations dataset")


In [ ]:
ds = load_dataset("carlomarxx/trilemma-of-truth", "city_locations")

print("train:", ds["train"].shape)
print("test:", ds["test"].shape)
print("validation:", ds["validation"].shape)

## 3. The event model and the workshop data

The second half of the workshop uses two things:

1. **[`carlomarxx/synthea-bert`](https://huggingface.co/carlomarxx/synthea-bert)**:  a
   BERT-style model trained on synthetic health records (~4 MB).
2. **[`carlomarxx/synthea-workshop-data`](https://huggingface.co/datasets/carlomarxx/synthea-workshop-data)**: 50,000 synthetic patients generated via [Synthea](https://synthetichealth.github.io/synthea/)
   (~19 MB).

About **23 MB** together, so this part is quick.

The cell below looks for local copies first and downloads from the Hugging Face Hub if it
does not find them. **On Colab there are no local copies, so the Hub is the only path**, which is
why those two repositories are public and need no token.

In [ ]:
import pathlib

MODEL_REPO = "carlomarxx/synthea-bert"
DATA_REPO = "carlomarxx/synthea-workshop-data"


def resolve(local_path, repo_id, repo_type="model"):
    """Prefer the copy in this repository; fall back to the Hub if it is not here."""
    if pathlib.Path(local_path).exists():
        return str(local_path), "already in this repository"
    from huggingface_hub import snapshot_download

    return snapshot_download(repo_id, repo_type=repo_type), f"downloaded from {repo_id}"


MODEL, model_source = resolve("../models/synthea-bert", MODEL_REPO)
DATA, data_source = resolve("../data/derived/workshop", DATA_REPO, "dataset")

print(f"event model:   {MODEL}\n               ({model_source})")
print(f"workshop data: {DATA}\n               ({data_source})")

Now load it, rather than just downloading it. The cell below rebuilds the architecture and loads the weights into it, which is what actually fails if something is wrong.

In [ ]:
import sys

# `event_bert.py` ships inside the model repo as well as living in `scripts/`, so
# this works with or without a checkout. `../scripts` is inserted last, so it wins
# when it exists and the Hub copy is only used on Colab.
sys.path.insert(0, MODEL)
sys.path.insert(0, "../scripts")

from event_bert import EventBertForMaskedLM

vocabulary = pd.read_csv(f"{DATA}/vocabulary.csv")
event_model = EventBertForMaskedLM.from_pretrained(
    MODEL, expected_vocab_size=len(vocabulary)
)

print(f"Loaded the event model")
print(f"  {sum(p.numel() for p in event_model.parameters()):,} parameters")
print(f"  {len(vocabulary)} tokens in the event vocabulary")
print(
    f"  {(vocabulary.kind == 'event').sum()} event types, "
    f"{(vocabulary.kind == 'background').sum()} background, "
    f"{(vocabulary.kind == 'special').sum()} special"
)

## 4. That's it

Everything the workshop needs is now in your local caches: the ~600 MB language model for the
first half, and the ~23 MB event model and data extract for the second. None of it will touch
the conference wifi on the day.